# TFM — 06. Predicción sobre datos nuevos (productivización)
**Autora:** Meritxell Abellan Collado

Este notebook demuestra la **productivización** del modelo: dado un Excel nuevo con el mismo formato que el original (p. ej. una futura publicación de datos del Ayuntamiento), se aplica automáticamente todo el proceso — limpieza, preprocesado, feature engineering y predicción — sin repetir manualmente ningún paso de los notebooks 01-04.

Todo el trabajo pesado vive en `utils/pipeline_produccion.py` (clase `PipelineAccidentes`), ya ajustado y guardado en `04_Modelizacion.ipynb`. Este notebook solo carga lo ya guardado y lo aplica — es, en esencia, la versión interactiva de `predict.py` (ejecutable también desde línea de comandos, ver ese script para el uso en producción real).

**Nota importante:** este flujo asume un Excel con el mismo formato *histórico* (con `LESIVIDAD` conocida por persona) — útil para revalidar el modelo sobre un nuevo lote de datos ya cerrado (p. ej. el año siguiente, una vez publicado). Para predecir un accidente individual en el momento del aviso (sin conocer aún la gravedad de nadie, el caso de uso real de la Pregunta 1), se usa una función distinta y más ligera — ver la última sección de este notebook.


## 1. Cargar el pipeline y el modelo ya entrenados

In [ ]:
import pandas as pd
import joblib

import sys
sys.path.append('..')
from src.config import cfg
from src.pipelines.pipeline_produccion import PipelineAccidentes

pipe = PipelineAccidentes.load(str(cfg.ruta(cfg.paths['models_dir']) / cfg.model['produccion']['pipeline']))
modelo_operativo = joblib.load(str(cfg.ruta(cfg.paths['models_dir']) / cfg.model['produccion']['modelo_operativo']))

print(f'Pipeline ajustado con {len(pipe.features_operativo)} features operativas.')
print(f'Categoría de referencia TIPO_VIA: {pipe.ref_via} | TIPO ACCIDENTE: {pipe.ref_acc}')


## 2. Cargar el Excel nuevo y predecir

In [ ]:
RUTA_EXCEL_NUEVO = '../data/raw/excel_nuevo.xlsx'  # sustituir por la ruta real

df_nuevo = pd.read_excel(RUTA_EXCEL_NUEVO)
print(f'{len(df_nuevo):,} registros de persona leídos de {RUTA_EXCEL_NUEVO}')

resultado = pipe.predecir(df_nuevo, modelo_operativo, tipo='operativo')
resultado.head(10)


## 3. Resumen de las predicciones

In [ ]:
print(f'Accidentes procesados: {len(resultado):,}')
print(f'Probabilidad media predicha de gravedad: {resultado["probabilidad_grave"].mean()*100:.2f}%')
print(f'Nº de accidentes con probabilidad > 50%: {(resultado["probabilidad_grave"] > 0.5).sum():,}')

# Si el Excel nuevo incluye GRAVE real (dato histórico ya cerrado), se puede
# validar el modelo sobre este nuevo lote:
if resultado['GRAVE'].notna().all():
    from sklearn.metrics import roc_auc_score
    auc = roc_auc_score(resultado['GRAVE'], resultado['probabilidad_grave'])
    print(f'\nAUC-ROC sobre este nuevo lote: {auc:.4f}')
    print('(comparar con el AUC-ROC de test original en 04_Modelizacion.ipynb, para detectar deriva del modelo)')


In [ ]:
resultado.to_csv('../data/processed/predicciones_nuevas.csv', index=False)
print('Predicciones guardadas en ../data/processed/predicciones_nuevas.csv')


## 4. Predicción de un accidente individual (caso de uso real: en el momento del aviso)

A diferencia de las secciones anteriores (que parten de un Excel completo con `LESIVIDAD` ya conocida), esta función recibe únicamente los datos disponibles **en el momento del aviso** — sin ningún dato de las personas implicadas — y devuelve directamente una probabilidad. Es el caso de uso real de la Pregunta 1 (apoyo a SAMUR / Policía Municipal / Emergencias 112).

*(pendiente de implementar — requiere construir una versión de `transform()` que no dependa de `agregar_flags_persona`/`construir_dataset_accidente`, ya que esos pasos asumen que ya se conoce el desenlace de cada persona implicada)*
